In [15]:
!pip install rapidfuzz unidecode cleanco


In [2]:
import re  # Used for working with regular expressions.
from dataclasses import dataclass  # For building simpler classes.
from pathlib import Path  # Not necessary at this point but keep it on the radar :)
from typing import Dict, List, Optional, Tuple  # Type hints for better readability.
from cleanco import prepare_default_terms, basename  # Cleanco functions.
import pandas as pd  # The old Classic.
from rapidfuzz import fuzz  # This will be our main engine for this task.
from unidecode import unidecode  # Word normalization.

In [3]:
#-----CONFIG------
# We will define the routes, main variables, and some measures.
# Defining them here makes it easy in case we need to make quick changes, and keeps better control.
@dataclass
class Config:
    # NOTE: Ensure these are converted to .csv or .xlsx in your Drive, as .gsheet files cannot be read directly by pandas.
    input_file: str = "/content/drive/MyDrive/Colab Notebooks/Text analytics.csv"
    output_file: str = "/content/drive/MyDrive/Colab Notebooks/Text analytics results.csv"

    column1: str = "Name_1"
    column2: str = "Name_2"

    # Fuzzy thresholds (Rapid Fuzz)
    UMBRAL_EXACT: float = 98.0
    UMBRAL_LIKELY: float = 95.0
    UMBRAL_POSSIBLE: float = 90.0

    # Gate behavior
    # The gate is supposed to prevent the comparison between 2 entities when they are from different market sectors.
    # There's no point in comparing these kinds of matches.
    ENABLE_TYPE_GATE: bool = True

    # At the beginning, this code was thought to be used with a fallback to sentence transformers,
    # but as I explained before, it seems too hard to add it, even when Fuzzy can help us a lot on this point.
    # Nevertheless, this trigger will help us to understand cases when a more robust transformer is needed.
    # If fuzzy score is below this, use embeddings.
    # EMBEDDINGS_TRIGGER: float = 90.0
    # Embedding thresholds
    # EMB_UMBRAL_LIKELY: float = 92.0
    # EMB_UMBRAL_POSSIBLE: float = 88.0
    # MODEL_NAME: str = "paraphrase-multilingual-MiniLM-L12-v2"
    # BATCH_SIZE: int = 32
    # DEVICE: str = "cuda"



In [4]:
# This MatchType will help us to determine the match classification.
class MatchType:
  EXACT = "EXACT MATCH"
  LIKELY = "LIKELY MATCH"
  POSSIBLE = "POSSIBLE MATCH"
  NO_MATCH = "NO MATCH"
  NO_DATA = "NO DATA"

# This part is very important, allowing you to add or delete as many EntityTypes (Market Sectors) as you want.
class EntityType:
  CORP = "CORPORATES"
  INSTRUMENT = "FINANCIAL_INSTRUMENT"
  MIXED = "MIXED_DEAL_CORP"
  OTHER = "OTHER"

In [5]:
# The TextNormalizer function will clean the name, remove suffixes, and classify the entity before scoring with Fuzzy.

class TextNormalizer:
    def __init__(self):
      # We are using Cleanco to obtain the legal suffixes; however, I noticed that cleanco doesn't cover all the suffixes I have in my data.
      # That's the main reason you need to prepare your data before working with any Python code.
      # I added my own dictionary for my convenience, but you can edit it freely.
      # There is no limit to the dictionary; everything can be tailored according to your business necessities.
        self.terms = prepare_default_terms()

        self.custom_legal_suffixes = {
            "s a de c v": EntityType.CORP,
            "s a":EntityType.CORP,
            "sab de cv": EntityType.CORP,
            "sofom": EntityType.CORP,
            "sofipo": EntityType.CORP
        }

        self.instrument_keywords = {
            'term loan': EntityType.INSTRUMENT,
            'notes': EntityType.INSTRUMENT,
            'bond': EntityType.INSTRUMENT,
            'fund': EntityType.INSTRUMENT,
            'etf': EntityType.INSTRUMENT,
            'serie': EntityType.INSTRUMENT,
            'tranche': EntityType.INSTRUMENT,
            'facility': EntityType.INSTRUMENT,
            'revolver': EntityType.INSTRUMENT,
            'clo': EntityType.INSTRUMENT,
            'cdo': EntityType.INSTRUMENT,
            'abs': EntityType.INSTRUMENT,
            'mbs': EntityType.INSTRUMENT,
            'warrant': EntityType.INSTRUMENT,
            'debenture': EntityType.INSTRUMENT,
            'preferred': EntityType.INSTRUMENT
        }

        self.public_entity_keywords = {
            'city of': EntityType.OTHER,
            'state of': EntityType.OTHER,
            'republic of': EntityType.OTHER,
            'gobierno': EntityType.OTHER,
            'municipio': EntityType.OTHER,
            'commonwealth': EntityType.OTHER,
            'county of': EntityType.OTHER,
            'province of': EntityType.OTHER,
            'district of': EntityType.OTHER,
            'authority of': EntityType.OTHER,
            'department of': EntityType.OTHER,
            'ministry of': EntityType.OTHER,
            'banco central': EntityType.OTHER,
            'central bank': EntityType.OTHER,
            'county': EntityType.OTHER
        }

    # These dictionaries are just examples in order to not make the code too long.

    # Before using the suffixes obtained from cleanco, the code will search your custom dictionary.
    # This allows you to personalize your dictionaries according to your needs.


    def normalize(self, text: str) -> tuple:
        if text is None or (isinstance(text, float) and pd.isna(text)):
            return "", "", "NO_DATA"

        text = unidecode(text)                          # Eliminar acentos
        text = text = re.sub(r'\s*&\s*', ' ', text)     # & → and
        text = re.sub(r'(\d+)-(\d+)', r'\1 \2', text)   # Deal IDs
        text = re.sub(r'[^\w\s]', ' ', text)            # Eliminar puntuación
        text = text.lower().strip()                     # Minúsculas
        text = re.sub(r'\s+', ' ', text)                # Espacios múltiples

        found_instrument = False
        found_public = False
        found_corp = False

        legal = ""
        public_keyword = ""
        instrument_keyword = ""

        for keyword, entity_type in self.instrument_keywords.items():

            if keyword in text:

              found_instrument = True
              instrument_keyword = keyword

        for keyword, entity_type in sorted(
            self.public_entity_keywords.items(),
            key=lambda x: len(x[0]),
            reverse=True
        ):

            if keyword in text and not public_keyword:

                found_public = True
                public_keyword = keyword


        for keyword, entity_type in sorted(
            self.custom_legal_suffixes.items(),
            key=lambda x: len(x[0]),
            reverse=True
        ):
            if keyword in text and not legal:

                found_corp = True
                legal = keyword

# Sorting by length helps prevent mismatches in the dictionaries.
# FALLBACK TO CLEANCO FOR THOSE SUFFIXES NOT PRESENT IN THE DICTIONARY

        base = basename(text, terms=self.terms, middle=False, last=True).strip()

        if base != text and not legal:

            legal = text.replace(base, '').strip()
            found_corp = True



        if found_corp and found_public:
            entity_type = EntityType.MIXED

        elif found_corp and found_instrument:
            entity_type = EntityType.MIXED

        elif found_corp:
            entity_type = EntityType.CORP

        elif found_instrument:
            entity_type = EntityType.INSTRUMENT

        elif found_public:
            entity_type = EntityType.OTHER

        else:
            entity_type = EntityType.OTHER

        if legal:
            base = base.replace(legal, '').strip()

        if public_keyword:
            base = base.replace(public_keyword, '').strip()

        if instrument_keyword:
            base = base.replace(instrument_keyword, '').strip()

        base = re.sub(r'\s+', ' ', base).strip()

        suffix = legal or public_keyword or instrument_keyword
        return base, suffix, entity_type


In [8]:
# GATE PASS
# Before using Fuzzy, we will determine if the entities are worth comparing according to their suffix and EntityType.
# Input file
df = pd.read_excel(config.input_file)

# Column1 Normalization
df['Name_1_Normalized'] = df[config.column1].apply(normalizer.normalize)
df['Name_1_Base'] = df['Name_1_Normalized'].apply(lambda x: x[0])
df['Name_1_Suffix'] = df['Name_1_Normalized'].apply(lambda x: x[1])
df['Name_1_Entity_Type'] = df['Name_1_Normalized'].apply(lambda x: x[2])
df.drop(columns=['Name_1_Normalized'], inplace=True)

# Column2 NORMALIZATION
df['Name_2_Normalized'] = df[config.column2].apply(normalizer.normalize)
df['Name_2_Base'] = df['Name_2_Normalized'].apply(lambda x: x[0])
df['Name_2_suffix'] = df['Name_2_Normalized'].apply(lambda x: x[1])
df['Name_2_Entity_Type'] = df['Name_2_Normalized'].apply(lambda x: x[2])
df.drop(columns=['Name_2_Normalized'], inplace=True)

# TYPE GATE
def type_gate(type1, type2):
    if config.ENABLE_TYPE_GATE == True:
        if type1 == type2:
            return True
        else:
            return False
    else:
        return True

df['GATE_PASS'] = df.apply(lambda row: type_gate(row['Name_1_Entity_Type'], row['Name_2_Entity_Type']), axis=1)


In [14]:
# MATCHING ENGINE
# Fuzzy Score

def calculate_fuzzy_score(text1, text2):
    return fuzz.token_sort_ratio(text1, text2)

# Why use token_sort_ratio?
# Even after normalization, some companies might retain words in different positions.
# Sorting them removes structural differences and focuses purely on whether the same words exist in both strings.

def classify_match(score):
    if score >= config.UMBRAL_EXACT:
        return MatchType.EXACT
    elif score >= config.UMBRAL_LIKELY:
        return MatchType.LIKELY
    elif score >= config.UMBRAL_POSSIBLE:
        return MatchType.POSSIBLE
    else:
        return MatchType.NO_MATCH

# 1. Initialize the column with a default float value
df['FUZZY_SCORE'] = 0.0

# 2. Calculate the real fuzzy score only for rows where GATE_PASS is True
df.loc[df['GATE_PASS'] == True, 'FUZZY_SCORE'] = df.loc[df['GATE_PASS'] == True].apply(
    lambda row: calculate_fuzzy_score(row['Name_1_Base'], row['Name_2_Base']), axis=1)

# 3. Classify based on the calculated scores
df['MATCH_TYPE'] = df.loc[df['GATE_PASS'] == True].apply(lambda row: classify_match(row['FUZZY_SCORE']), axis=1)
df.loc[df['GATE_PASS'] == False, 'MATCH_TYPE'] = MatchType.NO_MATCH

# Save results to the output file
df.to_excel(config.output_file, index=False)
